In [2]:
import random
import re
from nltk.translate.bleu_score import sentence_bleu
from nltk.corpus import wordnet
from pymystem3 import Mystem
import nltk

In [3]:
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [4]:
def split_string_by_punctuation(text):
    split_text = re.split('[.!?]', text)
    split_text = [sentence.strip() for sentence in split_text if sentence.strip()]
    return split_text

def swap_words(sentence):
    words = re.findall('\w+', sentence)
    swapped_words = random.sample(words, min(2, len(words)))
    for i, word in enumerate(swapped_words):
        synonyms = wordnet.synsets(word)
        if synonyms:
            words[words.index(word)] = random.choice(synonyms[0].lemma_names())
    return ' '.join(words)

def insert_words(sentence):
    words = re.findall('\w+', sentence)
    inserted_words = random.sample(words, min(2, len(words)))
    for i, word in enumerate(inserted_words):
        synonyms = wordnet.synsets(word)
        if synonyms:
            new_word = random.choice(synonyms[0].lemma_names())
            words.insert(words.index(word) + 1, new_word)
    return ' '.join(words)

def delete_words(sentence):
    words = re.findall('\w+', sentence)
    deleted_words = random.sample(words, min(2, len(words)))
    for i, word in enumerate(deleted_words):
        words.remove(word)
    return ' '.join(words)

def replace_words(sentence):
    words = re.findall('\w+', sentence)
    replaced_words = random.sample(words, min(2, len(words)))
    for i, word in enumerate(replaced_words):
        synonyms = wordnet.synsets(word)
        if synonyms:
            new_word = random.choice(synonyms[0].lemma_names())
            words[words.index(word)] = new_word
    return ' '.join(words)

def generate_augmented_text(text):
    augmented_sentences = []
    mystem = Mystem()
    text = split_string_by_punctuation(text)
    lemmas = list(map(mystem.lemmatize,text))
    for i in range(len(lemmas)):
      a = []
      for j in range(len(lemmas[i])):
          sentence = ' '.join(lemmas[i][:j+1])
          swapped_sentence = swap_words(sentence)
          inserted_sentence = insert_words(sentence)
          deleted_sentence = delete_words(sentence)
          replaced_sentence = replace_words(sentence)
          a.extend([swapped_sentence, inserted_sentence, deleted_sentence, replaced_sentence])
      augmented_sentences.append(a)
      a = []
    return augmented_sentences

In [56]:
def aug_paragraph(text, treshhold=0.7):
  augmented_text = generate_augmented_text(text)
  text = split_string_by_punctuation(text)
  a, b = [], []
  for i in range(len(augmented_text)):
    for j in range(len(augmented_text[i])):
      if sentence_bleu([text[i]], augmented_text[i][j], smoothing_function=nltk.translate.bleu_score.SmoothingFunction().method0) > treshhold:
          a.append(augmented_text[i][j])
    b.append(a)
    a = []
  return ' '.join([max(sentences, key=len) for sentences in b])

In [57]:
a = aug_paragraph('''In 323 BC, Alexander the Great died suddenly, leaving behind a huge empire that included a significant part of the Balkan Peninsula, the Aegean Islands, vast territories in Asia, including part of India, as well as Egypt. After a short period of turmoil, Alexander's feeble-minded brother Philip III Arrideus, as well as Alexander's newborn son by Roxana, were declared the new Macedonian kings.''')

/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 4-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 3-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_msg)
/usr/local/lib/python3.10/dist-packages/nltk/translate/bleu_score.py:552: UserWarning: 
The hypothesis contains 0 counts of 2-gram overlaps.
Therefore the BLEU score evaluates to 0, independently of
how many N-gram overlaps of lower order it contains.
Consider using lower n-gram order or use SmoothingFunction()
  warnings.warn(_

Идентификатор сервисного аккаунта
    ajevet6jnm8rrdht20am
Имя
    cf-translate-sa

Идентификатор ключа:
  ajesena1du0qq3tgi8j6
Ваш секретный ключ:
  AQVN3HXUFvga0PsJIma0TH6KHYutWuE3gEGzRNUZ

folderID на всякий случай
b1gp7jhf587t6mca8tam

In [58]:
import requests

IAM_TOKEN = 'AQVN3HXUFvga0PsJIma0TH6KHYutWuE3gEGzRNUZ'
folder_id = 'b1gp7jhf587t6mca8tam'
target_language = 'ru'
texts = a

body = {
    "targetLanguageCode": target_language,
    "texts": texts,
    "folderId": folder_id,
}

headers = {
    "Content-Type": "application/json",
    "Authorization": "Api-Key {0}".format(IAM_TOKEN)
}

response = requests.post('https://translate.api.cloud.yandex.net/translate/v2/translate',
    json = body,
    headers = headers
)

print(response.text)


{
 "translations": [
  {
   "text": "В 323 году до н.э. Александр Македонский умер, внезапно отказавшись от престола, оставив после себя огромную империю, которая включала значительную часть Балканского полуострова, острова Эгейского моря, обширные территории в Азии, включая часть Индии в качестве атомного номера33. сыновья Роксаны были объявлены новыми македонскими царями",
   "detectedLanguageCode": "en"
  }
 ]
}

